# LEGAL IMPRINT WAW Open Search Hackathon TUTORIAL

## Setup
 For this tutorial we use the Legal Imprint Dataset from the OpenWebSearch.eu project

The data, originally stored in parquet files, is converted into a duckdb for easier data handling.


#### PYTHON SETUP
We also provide a requirements.txt with all the necessary Python packages. With that, you can create a conda/mamba environment for the tutorial:

In [ ]:
mamba create -n waw_os_legal_imprint python=3.10  # mamba/conda depending on what you use
mamba activate waw_os_legal_imprint
pip install -r requirements.txt

# in jupyter
!pip install pandas geopandas duckdb

## Data Source

We use the [German Imprints Dataset](https://openwebindex.eu/corpora/4057d6a0-0bd9-11f1-89ba-02a47ca5d9fd) for this tutorial. This is a large-scale collection of websites annotated with the thematic allocation of the website and the geolocation of the extracted address of the imprint website. For the extraction of the two details an LLM pipeline was used. This dataset is provided in parquet format including the URL, html information of the main page, about page and imprint page.

## Data Extraction with DuckDB
For a better data handling in this tutorial, we used [DuckDB](https://duckdb.org/) to read the parquet files and extract relevant information. Afterward, we created a duckdb file with the relevant data for this tutorial. If further data need to be extracted or prefiltered this duckdb example can help to adjust for the requirements. In this tutorial we use the pyhton DuckDB package. The integration of the DuckDB client is also an option.

Load the following libraries

In [1]:
import duckdb
import pandas as pd
import geopandas as gpd

A DuckDB connection has to be established. It is connected to the duckdb file from the original parquet files.

In [6]:
# create the connection to the persistent database
con = duckdb.connect('legal_imprints_db.duckdb')

# duckdb  database overview
# Query the list of tables
tables = con.execute("SHOW TABLES").fetchall()

# Print table names
for table in tables:
    print(table)

('imprints_all',)


## Data Filtering

In [7]:
con.sql("""
CREATE TABLE imprints_with_coords AS SELECT
    host, topic, coords, address, region,
    TRY(JSON_EXTRACT(coords, '$.features[0].properties.osm_key'))::VARCHAR AS osm_key_raw,
    TRY(JSON_EXTRACT(coords, '$.features[0].properties.osm_id'))::VARCHAR AS osm_id_raw,
    TRY(JSON_EXTRACT(coords, '$.features[0].geometry.coordinates[0]'))::DOUBLE AS longitude_raw,
    TRY(JSON_EXTRACT(coords, '$.features[0].geometry.coordinates[1]'))::DOUBLE AS latitude_raw
FROM imprints_all
""")

BinderException: Binder Error: Referenced column "coords" not found in FROM clause!
Candidate bindings: "osm_id", "address", "longitude"

LINE 3:     host, topic, coords, address, region,
                         ^

## Data Saving
1. Convert the data into a pandas dataframe for further analysis
2. Save the filtered data a csv file for further analysis in a GIS System or just for storage

In [ ]:
# 1. Create pandas Dataframe
df = con.sql("SELECT * FROM imprints_with_coords").fetchdf()
print(df.head())

con.close()

# 2. Save to csv
con.sql("COPY imprints_with_coords TO 'imprints_with_coords.csv' (HEADER, DELIMITER ',')")

##DATA statistics
Explore the dataset, what columns are there and which content is given?
What could be useful information for your research question?

## Geospatial Analysis

Load the BKG vector dataset of all counties in Germany. This is then used to aggregate the point information on the county level and integrate the population and county size to compare the class distribution.

## DIY

Not it is your turn. Play around with certain filtering parameters, filtering for specific areas and topics and use the data for your scientific research question. You can also integrate your own (geospatial) data and synergistically analyze the data products.